# Spectral Guidance Example
## Eigenfunctions



In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt

from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from tqdm.auto import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import ExponentialLR
from diffusers import DDPMScheduler, DDIMScheduler

from examples.prior import *
from examples.denoiser import Denoiser
from examples.encoder import TimeConditionedEncoder
from spectral import whitening

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
prior = Circle(device=device)
#prior = Plane(device=device)
#prior = Disk(device=device)
#prior = Annulus(device=device)
prior_samples, prior_labels = prior.sample(batch_size=1500, return_labels=True)

plt.figure()
plt.scatter(prior_samples[:,0].cpu(), prior_samples[:,1].cpu(), 20, alpha=0.5, c=prior_labels.cpu() if prior_labels is not None else None, label="Prior", cmap="jet" if prior_labels is not None else None)
plt.axis("equal")
plt.title("Prior (Ground-Truth) Distribution")

## Train $f_\phi$

In [ ]:
num_eigenfunctions = 30
num_train_steps = 4000
batch_size = 4096
lr = 2e-4
ridge = 1e-3
eps = 1e-8

noise_scheduler = DDIMScheduler(
    beta_schedule="linear",
    beta_start=1e-4,
    beta_end=0.02,
    num_train_timesteps=1000,
    clip_sample=False,
)
noise_scheduler.set_timesteps(100)
timesteps = [int(t) for t in noise_scheduler.timesteps]

phi_encoder = TimeConditionedEncoder(in_dim=prior.dim, hidden_dim=128, t_dim=128, out_dim=num_eigenfunctions).to(device)
phi_encoder.train()

optimizer = AdamW(phi_encoder.parameters(), lr=lr)
scheduler = ExponentialLR(optimizer, gamma=0.9995)

eigenvalues_t = {}
pbar = tqdm(range(num_train_steps), desc="Training phi network")
for train_step in pbar:
    x0 = prior.sample(batch_size).to(device)
    t = np.random.choice(timesteps)
    t_batch = torch.full((x0.shape[0],), t, device=device)
    
    noise_a = torch.randn_like(x0)
    noise_b = torch.randn_like(x0)
    
    x_a = noise_scheduler.add_noise(x0, noise_a, t_batch)
    x_b = noise_scheduler.add_noise(x0, noise_b, t_batch)
    
    with torch.no_grad():
        phi_a = phi_encoder(x_a, t_batch)
        
    phi_b = phi_encoder(x_b, t_batch)
    
    mu, W = whitening(phi_a, ridge=ridge)
    
    phi_a_w = (phi_a - mu) @ W
    phi_b_w = (phi_b - mu) @ W
    
    phi_a_w = phi_a_w / (eps + phi_a_w.std(dim=0, keepdims=True)) # Renormalize
    phi_b_w = phi_b_w / (eps + phi_b_w.std(dim=0, keepdims=True)) # Renormalize

    eigenvalues = (phi_a_w * phi_b_w).mean(0)
    loss = (1.0 - eigenvalues.mean())
    eigenvalues_t[t] = eigenvalues.detach().cpu()
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    scheduler.step()
    pbar.set_postfix({'lr' : f"{optimizer.param_groups[0]['lr']:1.3e}"})

eigenvalues_timestamps = sorted(eigenvalues_t.keys()) # (T,)
eigenvalues_stacked = torch.stack([torch.sort(eigenvalues_t[t])[0] for t in eigenvalues_timestamps]) # (T, K)

colors = plt.cm.jet(np.linspace(0, 1, num_eigenfunctions))
plt.figure()
for k in range(num_eigenfunctions):
    plt.scatter(eigenvalues_timestamps, eigenvalues_stacked[:,k], 1, color=colors[k])
plt.xlabel("Diffusion Time")
plt.ylabel("Eigenvalue")
plt.title(f"Spectral Decay (Leading {num_eigenfunctions} Eigenfunctions)")

In [ ]:
phi_encoder.eval()
prior_x, prior_y = prior.sample(10_000, return_labels=True)

phi, mu, w = {}, {}, {}
for t in tqdm(timesteps):
    t_batch = torch.full((prior_x.shape[0],), t, device=device)
    noise = torch.randn_like(prior_x)
    xt = noise_scheduler.add_noise(prior_x, noise, t_batch)
    with torch.no_grad():
        phi_x = phi_encoder(xt, t_batch)
        mu[t], w[t] = whitening(phi_x, ridge=0.0)
        phi_x = (phi_x - mu[t]) @ w[t]
    phi[t] = torch.cat((torch.ones((len(phi_x), 1), device=device), phi_x), dim=-1)

## Visualize Eigenfunctions

In [ ]:
t_plot = 100
num_samples = 50*50
plot_eigenfunctions = range(8)

total_avg = 500
phi_encoder.eval()

if prior.name == "circle":
    angle = torch.linspace(0, 2*np.pi, num_samples, device=device)
    x0 = prior.radius * torch.stack([torch.cos(angle), torch.sin(angle)], dim=-1)
elif prior.name == "plane" :
    X, Y = torch.meshgrid(torch.linspace(prior.low, prior.high, int(np.sqrt(num_samples))), torch.linspace(prior.low, prior.high, int(np.sqrt(num_samples))))
    x0 = torch.stack((X.flatten(), Y.flatten()), dim=-1).to(device)
elif prior.name == "disk":
    n = int(np.sqrt(num_samples * 4 / np.pi))
    coords = torch.linspace(-prior.radius, prior.radius, n, device=device)
    X, Y = torch.meshgrid(coords, coords)
    pts = torch.stack((X.flatten(), Y.flatten()), dim=-1)
    mask = (pts[:, 0]**2 + pts[:, 1]**2) <= prior.radius**2
    x0 = pts[mask]
elif prior.name == "annulus":
    area_ratio = (prior.outer_radius**2 - prior.inner_radius**2) / prior.outer_radius**2
    n = int(np.sqrt(num_samples * 4 / (np.pi * area_ratio)))
    coords = torch.linspace(-prior.outer_radius, prior.outer_radius, n, device=device)
    X, Y = torch.meshgrid(coords, coords)
    pts = torch.stack((X.flatten(), Y.flatten()), dim=-1)
    r2 = pts[:, 0]**2 + pts[:, 1]**2
    mask = (r2 <= prior.outer_radius**2) & (r2 >= prior.inner_radius**2)
    x0 = pts[mask]
else:
    x0 = prior.sample(num_samples).to(device)

fig = plt.figure(figsize=(3 * len(plot_eigenfunctions), 3))
phi_samples = torch.zeros((total_avg, len(x0), num_eigenfunctions), device=device)
for k in range(total_avg):
    with torch.no_grad():
        noise = torch.randn_like(x0)
        t_batch = torch.full((x0.shape[0],), t_plot, device=device)
        xt = noise_scheduler.add_noise(x0, noise, t_batch)
        phi_t = phi_encoder(xt, torch.full((len(x0),), t_plot, device=device))
        phi_samples[k] += (phi_t - mu[t_plot]) @ w[t_plot]
        
psi = phi_samples.mean(0)
mupsi, wpsi = whitening(psi, 1e-10)
psi = (psi - mupsi) @ wpsi

vmax_abs = max(abs(psi[:, k].cpu()).max().item() for k in plot_eigenfunctions)
n = len(plot_eigenfunctions)
fig = plt.figure(figsize=(2.5 * n + 0.5, 2.5))
gs = GridSpec(1, n + 1, width_ratios=[1]*n + [0.07], wspace=0.05)

for j, k_plot in enumerate(plot_eigenfunctions):
    ax = fig.add_subplot(gs[0, j])

    if prior.name == "plane":
        im = ax.imshow(psi[:,k_plot].view(int(np.sqrt(num_samples)), int(np.sqrt(num_samples))).cpu(), vmin=-vmax_abs, vmax=vmax_abs, cmap='RdBu_r', interpolation="lanczos")
    elif prior.name == "circle":
        im = ax.scatter(x0[:,0].cpu(), x0[:,1].cpu(), s=30, c=psi[:, k_plot].cpu(), cmap='RdBu_r', vmin=-vmax_abs, vmax=vmax_abs, alpha=1.0)   
    else:
        im = ax.scatter(x0[:,0].cpu(), x0[:,1].cpu(), s=10, c=psi[:, k_plot].cpu(), cmap='RdBu_r', vmin=-vmax_abs, vmax=vmax_abs, alpha=1.0)   
    ax.set_aspect('equal')
    ax.set_title(rf"$\psi_{{ {k_plot+2} }}$", fontsize=20)
    ax.axis('off')

cbar_ax = fig.add_subplot(gs[0, -1])
fig.colorbar(im, cax=cbar_ax)
fig.savefig(f"./psi_{prior.name}_t={t_plot}.pdf", bbox_inches='tight', pad_inches=0)